# DATA CLEANING AND CHECKING

As discussed in the provided README file, the dataset was selected based on its ease of availablity, the quality of the data included and the fact it is very comprehensive, providing data from 1950 to 2024. However, I only chose to include data from 2014-2024. The 2014 F1 season was the first in which Hybrid engines were mandatory for all teams. I felt this was important to consider as, when considering performance, it simplifies things somewhat when all constructors are at least using the same type of engine. Narrowing the dataset down to this time period also solved another issue. As the structure of F1 race weekends has changed much since its beginning in 1946, more and more variables have been added to the dataset, which has ed to large parts of the dataset being left empty, for example the qualifying 'knockout' format was only introduced in 2006, meaning the q1, q2 and q3 columns are empty for every race before then. The structure has changed very little since 2014, meaning that the data I have chosen to include is much more comprehensive. This is improtant as it would not necessarily be fair to compare modern and hisotrical race data when only half of the data is present for historical races. 

The following is my data cleaning and checking processes, to ensure that the data I need is present, accurate and in a usable state. The tables provided by the dataset that I use are races, results, circuits, qualifying and status. 

**Races** contains information about each individual race weekend, such as the year, circuit, time and the status (finished, DNF, collision etc). 

**Results** contains all results for every race, including the driver, constructor, starting and finishing positions. 

**Circuits** contains information about each circuit, such as the name and location.

**Qualifying** contains the results of qualifying for each race, including q1, q2, q3 times and qualifying position.

**Status** contains status information referenced in races.

In [2]:
# Import libraries and necessary csv files
import pandas as pd

path = "../data/"

races = pd.read_csv(path + "races.csv", na_values=["\\N"])
results = pd.read_csv(path + "results.csv", na_values=["\\N"])
circuits = pd.read_csv(path + "circuits.csv", na_values=["\\N"])
quali = pd.read_csv(path + "qualifying.csv", na_values=["\\N"])
status = pd.read_csv(path + "status.csv", na_values=["\\N"])

In [3]:
# Drop unneeded car number 
#results = results.drop(columns=["number"])
# Sort results table starting from 2024 and descending, and then by position order ascending
results = results.sort_values(['raceId', 'positionOrder'], ascending=[False, True])
# Sense-check race ID year value
races.loc[races['raceId'] == 1144, ['raceId', 'year']]

,raceId,year
1124,1144,2024


In [4]:
# Show number of rows and columns in each table
print("races: ", races.shape, 
      "\nresults: ", results.shape,
      "\ncircuits: ", circuits.shape,
      "\nqualifying: ", quali.shape)

races:  (1125, 18) 
results:  (26759, 18) 
circuits:  (77, 9) 
qualifying:  (10494, 9)


By displaying the columns each table contains, I can understand the available data and identify relevant variables, as not every column will be necessary for the analysis.

In [5]:
# Show column names in races, results and qualifying tables
print("races columns: ", races.columns.tolist(),
      "\nresults columns: ", results.columns.tolist(),
      "\nquali columns: ", quali.columns.tolist())

races columns:  ['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time'] 
results columns:  ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId'] 
quali columns:  ['qualifyId', 'raceId', 'driverId', 'constructorId', 'number', 'position', 'q1', 'q2', 'q3']


Here, I filter the races table to only those from 2014-2024, as discussed above. The new hybrid dataframes contain the relevant information from each table between 2014-2024. RaceId is used to restrict results in the results and qualifying table as this is the unique identifier helpfully present in these tables, which allows me to select the relevant records.

In [6]:
# Filter races for 2014-2024 seasons (post introduction of mandatory Hybrid engine)
hybridEra = races[(races["year"] >= 2014) & (races["year"] <= 2024)]
# Filter results based on races present in filtered hybridEra races table
hybridResults = results[results["raceId"].isin(hybridEra["raceId"])]
#hybridResults
hybridQuali = quali[quali["raceId"].isin(hybridEra["raceId"])]
hybridQuali

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
5857,5860,900,1,131,44,1,1:31.699,1:42.890,1:44.231
5858,5861,900,817,9,3,2,1:30.775,1:42.295,1:44.548
5859,5862,900,3,131,6,3,1:32.564,1:42.264,1:44.595
5860,5863,900,825,1,20,4,1:30.949,1:43.247,1:45.745
5861,5864,900,4,6,14,5,1:31.388,1:42.805,1:45.819
...,...,...,...,...,...,...,...,...,...
10489,10547,1144,848,3,23,16,1:23.821,NaN,NaN
10490,10548,1144,855,15,24,17,1:23.880,NaN,NaN
10491,10549,1144,1,131,44,18,1:23.887,NaN,NaN
10492,10550,1144,861,3,43,19,1:23.912,NaN,NaN


In [7]:
# Confirm hybridEra table contains correct years and show number of races per year
print("min year: ", hybridEra["year"].min(),
      "\nmax year: ", hybridEra["year"].max(),
      "\nraces per year: ", hybridEra["year"].value_counts().sort_index())


min year:  2014 
max year:  2024 
races per year:  year
2014    19
2015    19
2016    21
2017    20
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
Name: count, dtype: int64


Here, I inspect how many null values there are in each column in each new table. This is so I can ensure I have enough usable data and that there are not unexplained missing values. 

In [8]:
# Show number of null datapoints in new races, results and qualifying tables 
print("races NaN: ", hybridEra.isnull().sum(),
      "\nresults NaN: ", hybridResults.isnull().sum(),
      "\nqualifying NaN: ", hybridQuali.isnull().sum())

races NaN:  raceId           0
year             0
round            0
circuitId        0
name             0
date             0
time             0
url              0
fp1_date       138
fp1_time       160
fp2_date       138
fp2_time       160
fp3_date       156
fp3_time       175
quali_date     138
quali_time     160
sprint_date    210
sprint_time    213
dtype: int64 
results NaN:  resultId              0
raceId                0
driverId              0
constructorId         0
number                0
grid                  0
position            721
positionText          0
positionOrder         0
points                0
laps                  0
time               2219
milliseconds       2219
fastestLap          217
rank                  0
fastestLapTime      217
fastestLapSpeed     217
statusId              0
dtype: int64 
qualifying NaN:  qualifyId           0
raceId              0
driverId            0
constructorId       0
number              0
position            0
q1                 51
q

The position columns contains the numerical finishing position of each driver in a race. If a driver does not recieve a conventional finishing position, retired, or recieved a penalty, that is recorded in position text. By exploring the contents of the position text column, I am able to determine that all the missing positions have an appropriate position text value, and as such are still accounted for in some capacity. For exmple, out of the 721 records with missing positions, I can see that 680 retired, 28 withdrew and 11 were disqualified. As my research question examines the relationship between qualifying and finishing position, where there is no finishing position these records will be excluded from the final analysis.

In [9]:
# Create table of rows with no finishing position 
missingPos = hybridResults[hybridResults["position"].isnull()]

print(len(missingPos)) # Get the length of this table
# Show distribution of each value in position text, to see reasons for missing finishing
# positions to confirm they are accounted for 
print(missingPos["positionText"].value_counts(), missingPos["positionOrder"].describe())

721
positionText
R    680
W     28
D     11
E      1
N      1
Name: count, dtype: int64 count    721.000000
mean      18.690707
std        1.758325
min       12.000000
25%       18.000000
50%       19.000000
75%       20.000000
max       22.000000
Name: positionOrder, dtype: float64


Here, I can see the number of times each grid position shows up in the table. The grid position is the position that a driver started the race in. In this dataset, a grid position of 0 means that the driver started in the pit lane, and therefore did not have an official starting position. However, again as my research question involves qualifying and finishing position, it does not consider starting position and thus grid = 0 was not used as en exclusionary criteria. Furthermore, there are only 79 records that have grid = 0, which should not signifciantly impact or sway the analysis anyway. 

In [10]:
# Show number of each starting grid position in results table
print(hybridResults["grid"].value_counts().sort_index())

grid
0      79
1     228
2     226
3     228
4     228
5     228
6     228
7     227
8     227
9     228
10    226
11    226
12    228
13    226
14    227
15    226
16    226
17    230
18    222
19    216
20    176
21     36
22     34
Name: count, dtype: int64


The dataset does not require any editing or manipulation as all the data I require is already clean and usable. In this notebook I decided on what data to use, and determined that the number of records with missing values that will be excluded is reasonable, and no further cleaning needs to be carried out. In the next notebook, I process the tables to do further checks to create a final dataframe to be used for the analysis. 